In [ ]:
import sys
import os
import json
from datetime import datetime, timedelta

# Add backend directory to sys.path to allow importing agents
current_dir = os.getcwd()
backend_dir = os.path.abspath(os.path.join(current_dir, '..'))
if backend_dir not in sys.path:
    sys.path.append(backend_dir)

from agents.verification_agent import VerificationAgent

In [ ]:
def process_single_date(date_str):
    """
    Processes valid news for a single date string (YYYYMMDD).
    Input: final_news_verified_{date_str}.json
    Output: final_news_reverified_{date_str}.json
    """
    input_path = f"final_news_verified_{date_str}.json"
    output_path = f"final_news_reverified_{date_str}.json"
    
    if not os.path.exists(input_path):
        print(f"Skipping {date_str}: Input file {input_path} not found.")
        return None
        
    print(f"\n=== Processing Date: {date_str} ===")
    print(f"Loading input: {input_path}")
    
    # 1. Load Original Data
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            original_data = json.load(f)
    except Exception as e:
        print(f"Error reading {input_path}: {e}")
        return None

    # 2. Run VerificationAgent (Strict Tags + Translations)
    print("Initializing VerificationAgent...")
    agent = VerificationAgent()
    
    reverified_data = []
    print(f"Verifying {len(original_data)} articles...")
    
    for i, article in enumerate(original_data):
        content = article.get('raw_content') or article.get('content') or article.get('summary')
        try:
            # print(f"  - Processing {i+1}/{len(original_data)}: {article.get('id')}")
            processed = agent.verify_and_format(article, content)
            if processed:
                reverified_data.append(processed)
            else:
                print(f"  - Failed to verify: {article.get('id')}")
        except Exception as e:
            print(f"  - Error on {article.get('id')}: {e}")

    # 3. Merge Missing Metadata & Restore IDs
    print("Merging metadata and restoring IDs...")
    
    final_list = []
    updated_count = 0
    restored_ids = 0
    
    # We iterate by index, assuming order is substantially preserved or we match by content.
    # Since VerificationAgent processes list sequentially, order should match.
    
    for i, item in enumerate(reverified_data):
        if i >= len(original_data):
            break
            
        original_item = original_data[i]
        
        # Restore ID
        if 'id' in original_item and item.get('id') != original_item['id']:
            item['id'] = original_item['id']
            restored_ids += 1
            
        # Restore Fields
        fields_to_check = ['source', 'sourceUrl', 'publishedDate', 'likes', 'viewCount', 'shareCount']
        item_updated = False
        
        for key in fields_to_check:
            val = item.get(key)
            if val is None or val == "None" or val == "":
                orig_val = original_item.get(key)
                if orig_val is not None and orig_val != "None" and orig_val != "":
                    item[key] = orig_val
                    item_updated = True
        
        final_list.append(item)
        if item_updated:
            updated_count += 1
            
    print(f"Restored {restored_ids} IDs.")
    print(f"Merged metadata for {updated_count} items.")
    
    # 4. Save Output
    print(f"Saving to {output_path}...")
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(final_list, f, indent=2, ensure_ascii=False)
        
    print(f"Completed {date_str}.")
    return output_path

def process_news_by_date_range(start_date, end_date):
    """
    Iterates through a date range (inclusive) and processes news for each day.
    Format: YYYY-MM-DD
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    current = start
    while current <= end:
        date_str = current.strftime("%Y%m%d")
        process_single_date(date_str)
        current += timedelta(days=1)


In [ ]:
# Example Usage
if __name__ == "__main__":
    # Set your date range here
    START_DATE = "2026-01-30"
    END_DATE = "2026-01-30"
    
    process_news_by_date_range(START_DATE, END_DATE)